In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from preprocessing import build_preprocessor, load_and_split

In [ ]:
X_train, X_test, y_train, y_test = load_and_split("../data/student-mat.csv")
X_test.head()

,G1,G2,failures,absences,higher,Medu,Fedu,goout,traveltime,age,studytime
78,8,8,3,2,no,2,1,1,2,17,1
371,14,12,0,3,no,1,2,3,3,18,1
248,3,5,1,8,yes,3,3,3,1,18,2
55,8,9,0,8,yes,2,1,4,1,16,2
390,9,9,2,11,yes,2,2,4,1,20,2


In [5]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessing", build_preprocessor()),
        ("model", model)
    ])

    # CV
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    cv_r2 = cv_scores.mean()

    # Train + Test
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    test_r2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)

    

    results.append({
        "Model": name,
        "CV R2": round(cv_r2, 4),
        "Test R2": round(test_r2, 4),
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4)
    })

df_results = pd.DataFrame(results).sort_values(by="Test R2", ascending=False)
display(df_results)

,Model,CV R2,Test R2,RMSE,MAE
1,Random Forest,0.8834,0.8488,1.7607,1.0815
2,Gradient Boosting,0.8763,0.8403,1.8097,1.1105
0,Linear Regression,0.8380,0.7690,2.1763,1.4274
